# Lesson 6.2 — VPC Connectivity: Peering, PrivateLink & Transit Gateway

**Level: Intermediate** | **Exam:** SOA-C03 Domain 5, Task 5.1 (Skill 5.1.2); SAP-C02 Domain 1, Task 1.1

## Learning Objectives
- Configure VPC peering for cross-VPC communication
- Use VPC Endpoints (Gateway and Interface) to access AWS services privately
- Design hub-and-spoke network topology with AWS Transit Gateway
- Implement PrivateLink for private service access across accounts

_Reference: [VPC Peering](https://docs.aws.amazon.com/vpc/latest/peering/) | [Transit Gateway](https://docs.aws.amazon.com/vpc/latest/tgw/) | [PrivateLink](https://docs.aws.amazon.com/vpc/latest/privatelink/)_

## Connectivity Options Comparison

| Option | Use Case | Transitive? | Cost |
|--------|---------|-------------|------|
| **VPC Peering** | Connect 2 VPCs (same/different account/region) | No | Data transfer only |
| **VPC Endpoint (Gateway)** | S3, DynamoDB — free, within AWS | N/A | Free |
| **VPC Endpoint (Interface)** | Other AWS services via PrivateLink | N/A | $0.01/hr + data |
| **Transit Gateway** | Hub-and-spoke for many VPCs | Yes | $0.05/hr per attachment |
| **VPN (Site-to-Site)** | On-premises to AWS | No | $0.05/hr per tunnel |
| **Direct Connect** | Dedicated physical line to AWS | Yes (with DX Gateway) | Physical port cost |

## VPC Peering — Rules
- Peering connections are **not transitive**:
  ```
  A ↔ B, B ↔ C does NOT mean A can talk to C
  (must add A ↔ C peering explicitly)
  ```
- CIDR ranges must NOT overlap (e.g., both VPCs on 10.0.0.0/16 — won't work)
- Cross-region peering supported, but higher data transfer cost
- Must update route tables on BOTH sides

## Transit Gateway — Hub and Spoke
```
        Transit Gateway
        ┌─────────────────┐
        │  TGW Route Table │
        └─────┬───┬───┬───┘
              │   │   │
            VPC-A VPC-B VPC-C
            /16   /16   /16
            ↑
          VPN/Direct Connect (on-prem)
```
Transit Gateway is transitive — VPC-A can reach VPC-B and VPC-C without direct peering.

## VPC Endpoints
- **Gateway Endpoints**: S3 and DynamoDB only. Free. Traffic stays on AWS network.
- **Interface Endpoints** (PrivateLink): All other services. Creates an ENI in your subnet.

```python
# Without VPC endpoint: S3 traffic goes out internet gateway → public internet → AWS S3
# With Gateway endpoint: S3 traffic routes through AWS backbone — private, no internet needed
```

## PrivateLink
Exposes a service in VPC-A to consumers in VPC-B **without peering**:
```
Consumer VPC ──→ Interface Endpoint (NLB IP) ──→ Service Provider VPC
(VPC-B)           (your subnet)                   (VPC-A, hidden)
```
Used by: AWS services, marketplace services, internal shared services across accounts.

In [ ]:
import boto3, json

ec2 = boto3.client('ec2', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# List existing VPC Endpoints
endpoints = ec2.describe_vpc_endpoints()['VpcEndpoints']

if endpoints:
    print(f'VPC Endpoints ({len(endpoints)}):')
    for ep in endpoints:
        print(f'  {ep["VpcEndpointId"]:25} Type: {ep["VpcEndpointType"]:10} Service: {ep["ServiceName"]}')
else:
    print('No VPC endpoints found.')
    print('\nCreate an S3 Gateway Endpoint (free, recommended for all VPCs):')
    print('ec2.create_vpc_endpoint(')
    print('    VpcId="vpc-xxx",')
    print('    ServiceName="com.amazonaws.us-east-1.s3",')
    print('    VpcEndpointType="Gateway",')
    print('    RouteTableIds=["rtb-xxx"]  # Private and public route tables')
    print(')')

In [ ]:
# Compare VPC connectivity topologies

topologies = {
    'VPC Peering (2 VPCs)': {
        'connections': 1,
        'transitive': False,
        'n_vpc_connections': lambda n: n*(n-1)//2,  # O(n^2)
        'n_vpcs': 2,
        'best_for': '2-3 VPCs with simple requirements'
    },
    'VPC Peering (10 VPCs)': {
        'connections': 45,  # 10*9/2
        'transitive': False,
        'n_vpcs': 10,
        'best_for': 'NOT recommended — complex, management overhead'
    },
    'Transit Gateway (10 VPCs)': {
        'connections': 10,  # 10 attachments to 1 TGW
        'transitive': True,
        'n_vpcs': 10,
        'best_for': 'Enterprise scale — centralized routing, VPN, Direct Connect'
    }
}

print('VPC Connectivity Topology Comparison:\n')
for name, t in topologies.items():
    print(f'{name}:')
    print(f'  Peering/Attachment connections: {t["connections"]}')
    print(f'  Transitive routing: {t["transitive"]}')
    print(f'  Best for: {t["best_for"]}')
    print()

print('Formula: VPC Peering for N VPCs requires N*(N-1)/2 peering connections')
print('Example: 10 VPCs = 45 peerings vs 10 TGW attachments')

In [ ]:
# Check for Transit Gateways in the account
try:
    tgws = ec2.describe_transit_gateways()['TransitGateways']
    if tgws:
        print(f'Transit Gateways ({len(tgws)}):')
        for tgw in tgws:
            print(f'  {tgw["TransitGatewayId"]}  {tgw["State"]}  {tgw["Description"]}')
            # List attachments
            attachments = ec2.describe_transit_gateway_attachments(
                Filters=[{'Name': 'transit-gateway-id', 'Values': [tgw['TransitGatewayId']]}]
            )['TransitGatewayAttachments']
            for att in attachments:
                print(f'    Attachment: {att["TransitGatewayAttachmentId"]} ({att["ResourceType"]}) {att["State"]}')
    else:
        print('No Transit Gateways found.')
        print('\nTGW is cost-effective when you have 3+ VPCs to connect.')
except Exception as e:
    print(f'TGW: {e}')

## Exercise

In [ ]:
# TODO: For each scenario, recommend the connectivity solution and calculate cost

scenarios = [
    {
        'description': '2 VPCs in same account, same region. Need to share RDS instance.',
        'vpcs': 2, 'cross_account': False, 'on_prem': False,
        'recommendation': 'TODO',  # VPC Peering / TGW / PrivateLink / VPC Endpoint
        'reason': 'TODO'
    },
    {
        'description': '15 spoke VPCs (dev accounts) need to access shared services VPC + on-prem.',
        'vpcs': 15, 'cross_account': True, 'on_prem': True,
        'recommendation': 'TODO',
        'reason': 'TODO'
    },
    {
        'description': 'EC2 in private subnet needs to access S3 without internet.',
        'vpcs': 1, 'cross_account': False, 'on_prem': False,
        'recommendation': 'TODO',
        'reason': 'TODO'
    },
]

for s in scenarios:
    print(f"Scenario: {s['description']}")
    print(f"  Recommendation: {s['recommendation']}")
    print(f"  Reason: {s['reason']}")
    print()

## Summary
- **VPC Peering**: Simple, cheap (data transfer only), but NOT transitive. Becomes complex at 5+ VPCs.
- **Transit Gateway**: Hub-and-spoke for 3+ VPCs. Transitive, supports VPN/Direct Connect attachment. $0.05/hr per attachment.
- **Gateway Endpoints** (S3/DynamoDB): Free, route traffic over AWS backbone — always add these!
- **Interface Endpoints** (PrivateLink): $0.01/hr per AZ — use for services that must stay private
- **Rule of thumb**: ≤2 VPCs = Peering; 3+ VPCs = Transit Gateway

---
**Next:** [Lesson 6.3 — Route 53: Routing Policies, Health Checks & DNS Firewall](lesson-6.3-route53-dns.ipynb)